# Bearing Inspection: From Metrics to Decisions

You're an ML engineer at a rail maintenance company. Train axle bearings
develop faults gradually through fatigue, cracking, or pitting. Caught early,
it's a scheduled replacement. Missed, it's an axle failure while the train is
in service.

Your team has two candidate computer vision models. Both classify bearing
inspection images as **defect** or **no defect**. Both were evaluated on the
same 500-image holdout set. Your job is to recommend which one to deploy and
at what decision threshold.

In this lab you'll compare the two models using confusion matrices and
precision-recall curves, explore how the decision threshold shifts the balance
between catching defects and raising false alarms, and build a deployment
recommendation you can justify.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, precision_recall_curve,
)

## Loading the data

The dataset contains pre-computed evaluation results for 500 bearing
inspection images. Of these, 100 contain confirmed defects. Each row records
the true label and both models' predicted probability that the image contains
a defect.

In [ ]:
df = pd.read_csv("bearings.csv")

print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df["true_label"].value_counts().rename({0: "No defect", 1: "Defect"}))
print(f"\nDefect rate: {df['true_label'].mean():.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["true_label"].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_xticklabels(["No defect", "Defect"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Class distribution in holdout set")
plt.tight_layout()
plt.show()

## Metrics at the default threshold

Both models output a probability between 0 and 1. At the default threshold of
0.5, any image scoring 0.5 or above is flagged as a defect.

In [ ]:
y = df["true_label"].values
prob_a = df["model_a_prob"].values
prob_b = df["model_b_prob"].values

pred_a = (prob_a >= 0.5).astype(int)
pred_b = (prob_b >= 0.5).astype(int)

metrics = pd.DataFrame({
    "Model A": {
        "Accuracy": accuracy_score(y, pred_a),
        "Precision": precision_score(y, pred_a),
        "Recall": recall_score(y, pred_a),
        "F1": f1_score(y, pred_a),
    },
    "Model B": {
        "Accuracy": accuracy_score(y, pred_b),
        "Precision": precision_score(y, pred_b),
        "Recall": recall_score(y, pred_b),
        "F1": f1_score(y, pred_b),
    },
})

metrics.round(3)

### Confusion matrices

The summary metrics hide the raw counts. Pay attention to the **false
negatives** (FN): those are defective bearings the model cleared for service.

In [ ]:
cm_a = confusion_matrix(y, pred_a)
cm_b = confusion_matrix(y, pred_b)

def cm_table(cm, name):
    tn, fp, fn, tp = cm.ravel()
    return pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual: no defect", "Actual: defect"],
        columns=["Pred: no defect", "Pred: defect"],
    )

print("Model A")
display(cm_table(cm_a, "Model A"))
print(f"\nModel B")
display(cm_table(cm_b, "Model B"))

## Which errors matter more?

Both models have high accuracy, but they make different kinds of mistakes.

A **false positive** means a healthy bearing gets flagged. An engineer inspects
it, finds nothing wrong, and sends it back into service. That costs time and
money.

A **false negative** means a defective bearing gets cleared. It goes back into
service and continues to degrade. If the defect is serious enough, the bearing
fails while the train is running.

These are not equivalent mistakes. In the cancer screening lab you saw the same
tension: missing a cancer is worse than an unnecessary biopsy. Here, the stakes
are physical infrastructure and passenger safety.

## The precision-recall trade-off

The metrics above assume a threshold of 0.5, but there's nothing sacred about
that number. Lowering the threshold catches more defects (higher recall) at the
cost of more false alarms (lower precision). The precision-recall curve shows
this trade-off across every possible threshold.

In [ ]:
prec_a, rec_a, _ = precision_recall_curve(y, prob_a)
prec_b, rec_b, _ = precision_recall_curve(y, prob_b)

pa_05 = precision_score(y, pred_a)
ra_05 = recall_score(y, pred_a)
pb_05 = precision_score(y, pred_b)
rb_05 = recall_score(y, pred_b)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(rec_a, prec_a, color="#2a6ebb", linewidth=2, label="Model A")
ax.plot(rec_b, prec_b, color="#e67e22", linewidth=2, label="Model B")

ax.scatter([ra_05], [pa_05], color="#2a6ebb", s=80, zorder=5)
ax.scatter([rb_05], [pb_05], color="#e67e22", s=80, zorder=5)

ax.annotate(f"A at 0.5\n({pa_05:.0%} prec, {ra_05:.0%} rec)",
            xy=(ra_05, pa_05), xytext=(ra_05 - 0.25, pa_05 - 0.1),
            fontsize=8, color="#2a6ebb",
            arrowprops=dict(arrowstyle="->", color="#2a6ebb"))
ax.annotate(f"B at 0.5\n({pb_05:.0%} prec, {rb_05:.0%} rec)",
            xy=(rb_05, pb_05), xytext=(rb_05 - 0.2, pb_05 - 0.12),
            fontsize=8, color="#e67e22",
            arrowprops=dict(arrowstyle="->", color="#e67e22"))

ax.set_xlabel("Recall (fraction of defects caught)")
ax.set_ylabel("Precision (fraction of flags that are real)")
ax.set_title("Precision-recall curves")
ax.legend()
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Every point on these curves is a possible deployment configuration. Model B's
curve sits above and to the right of Model A's for most of the range, meaning
it can achieve higher recall at the same precision, or higher precision at the
same recall.

The steep drop at the right-hand end is where the last few defects look almost
identical to healthy bearings. No amount of threshold tuning will catch them
cleanly. If those cases matter, you need a better model, not a lower threshold.

## Exploring the threshold

Change `THRESHOLD` in the cell below and re-run it to see how the metrics
shift for Model B.

**Task:** Find a threshold where Model B roughly matches Model A's default
precision (~0.94). What happens to recall?

In [ ]:
# Change this value and re-run
THRESHOLD = 0.50  # try values between 0.30 and 0.90

pred_b_t = (prob_b >= THRESHOLD).astype(int)

print(f"Model B at threshold {THRESHOLD}")
print(f"  Precision : {precision_score(y, pred_b_t):.3f}   (Model A default: {precision_score(y, pred_a):.3f})")
print(f"  Recall    : {recall_score(y, pred_b_t):.3f}   (Model A default: {recall_score(y, pred_a):.3f})")
print()
cm_bt = confusion_matrix(y, pred_b_t)
cm_ad = confusion_matrix(y, pred_a)
print(f"  Missed defects : {cm_bt[1, 0]}   (Model A default: {cm_ad[1, 0]})")
print(f"  False alarms   : {cm_bt[0, 1]}   (Model A default: {cm_ad[0, 1]})")

## Putting the numbers in context

Metrics on a holdout set are abstract. The cell below translates them into
missed defects per week on a busy route processing 2,000 inspection images.

In [ ]:
IMAGES_PER_WEEK = 2000
defect_rate = y.mean()
expected_defects = IMAGES_PER_WEEK * defect_rate

fn_rate_a = confusion_matrix(y, pred_a)[1, 0] / y.sum()
fn_rate_b = confusion_matrix(y, pred_b)[1, 0] / y.sum()

missed_a = expected_defects * fn_rate_a
missed_b = expected_defects * fn_rate_b

print(f"At {IMAGES_PER_WEEK:,} images per week:")
print(f"  Expected defects per week          : {expected_defects:.0f}")
print()
print(f"  Model A missed defects per week    : {missed_a:.1f}")
print(f"  Model B missed defects per week    : {missed_b:.1f}")
print()
print(f"  Extra missed defects/week with A   : {missed_a - missed_b:.1f}")
print(f"  Extra missed defects/year with A   : {(missed_a - missed_b) * 52:.0f}")

## Write your recommendation

Open `recommendation.md` in the file browser (left panel). It contains a short
template for your deployment recommendation. Fill in each section based on what
you've found in this notebook.

Be specific. For the questions about who is affected, think about a real person
in a real situation, not a general category. For the accountability question,
consider whether the answer changes depending on whether the failure was
foreseeable.

## Validate your recommendation

Run the cell below to check that each section of your recommendation has
enough detail. Every H2 section needs at least 15 words.

In [ ]:
%pip install -q mistune
from utils import validate_recommendation

validate_recommendation()